# Conservation laws

Some systems obey an exact algebraic invariant: a quantity that *must* stay
constant whatever the trajectory does. `ode_filters` can enforce it during the
solve, without leaving the public `gaussian_filter` API.

The mechanism is the one the solver already uses. A probabilistic solver treats
the ODE itself as a *measurement*: at each step it asks how badly the current
guess violates `dx/dt = f(x, t)` and corrects the Gaussian posterior
accordingly. A conservation law is just *another* measurement, folded into the
same Kalman update.

(ODE: ordinary differential equation; IVP: the initial value problem of solving
one forward from a known start. The filter is an extended Kalman filter, and the
Rauch–Tung–Striebel (RTS) smoother refines its result after the forward pass.)

We import the fixed-grid filter, the RTS smoother, the IWP prior, and the
Taylor-mode initializer from the top level, and `Conservation` from the
`ode_filters.information` submodule. The final lines set the shared
documentation plot style.

In [ ]:
import logging

import jax
import jax.numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from tueplots import bundles, cycler
from tueplots.constants.color import palettes

from ode_filters import (
    IWP,
    ODEInformation,
    gaussian_filter,
    rts_smoother,
    taylor_mode_initialization,
)
from ode_filters.information import Conservation

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(bundles.tue_ai_beamer())
plt.rcParams.update(cycler.cycler(color=palettes.paultol_bright))
set_matplotlib_formats("svg")  # vector figures stay sharp at any size
# Fall back to the default font quietly where Roboto Condensed is not installed.
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
REFERENCE = palette[6]  # grey: exact solutions and other references

## An epidemic that conserves its population

The classic example is an SIR epidemic model, where the population splits into
**S**usceptible, **I**nfected, and **R**ecovered fractions. People only ever move
S → I → R, so the total `S + I + R` is conserved — it should equal 1 for all
time.

A plain ODE solve does not *know* this. Each step introduces a tiny numerical
error, and those errors accumulate, so the total slowly drifts away from 1. We
can pin it down by handing the solver the invariant as a noiseless
pseudo-observation.

Here is the SIR vector field. Susceptibles become infected at rate
`beta * S * I`; infected recover at rate `gamma * I`. We start with a 1%
infected fraction.

In [ ]:
beta_sir = 0.5
gamma_sir = 0.1


def vf_sir(x, *, t):
    """SIR epidemic model with state x = [S, I, R]."""
    S, I, _ = x  # noqa: E741 (R does not enter the dynamics)
    return np.array(
        [
            -beta_sir * S * I,
            beta_sir * S * I - gamma_sir * I,
            gamma_sir * I,
        ]
    )


x0_sir = np.array([0.99, 0.01, 0.0])
tspan_sir = (0.0, 100.0)
N_sir = 100
ts_sir = np.linspace(*tspan_sir, N_sir + 1)

prior_sir = IWP(q=2, d=3, Xi=1.0 * np.eye(3))
mu_0_sir, P0_sqr_sir = taylor_mode_initialization(vf_sir, x0_sir, q=2)

A `Conservation(A, p)` object enforces the *linear* invariant `A @ x = p`
as a noiseless pseudo-observation: at every step the filter additionally
corrects toward states that satisfy it exactly. For the population sum we want
`[1, 1, 1] @ [S, I, R] = 1`, so `A` is a single row of ones and `p` is `[1.0]`.

We attach it through the `constraints=` argument of `ODEInformation`. Because a
conservation law is *always active* and has a fixed shape, it rides along inside
the ODE-information model — unlike data, which exist only at some times and enter
as a separate observation model (see the
[Inference from data](../parameter-estimation/) tutorials).

In [ ]:
A_sum = np.array([[1.0, 1.0, 1.0]])
p_sum = np.array([1.0])
conservation = Conservation(A_sum, p_sum)

# Two measurement models: the plain ODE, and the ODE plus the invariant.
measure_free = ODEInformation(vf_sir, prior_sir.E0, prior_sir.E1)
measure_cons = ODEInformation(
    vf_sir, prior_sir.E0, prior_sir.E1, constraints=[conservation]
)

We solve the IVP both ways — once without the constraint, once with it —
and smooth each forward pass with the RTS smoother. Everything uses the default
(`"dynamic"`) calibration; here we focus on the mean trajectory and its
invariant rather than the band.

In [ ]:
def solve(measure):
    result = gaussian_filter(
        mu_0_sir, P0_sqr_sir, prior_sir, measure, np.linspace(*tspan_sir, N_sir + 1)
    )
    m_smooth, _ = rts_smoother(result)
    return np.array(m_smooth)


m_free = solve(measure_free)
m_cons = solve(measure_cons)

# The conserved quantity along each trajectory, and its drift from 1.
total_free = m_free[:, :3].sum(axis=1)
total_cons = m_cons[:, :3].sum(axis=1)
resid_free = np.abs(total_free - 1.0)
resid_cons = np.abs(total_cons - 1.0)

On a logarithmic axis we plot how far `S + I + R` strays from 1. Without the
constraint the total drifts to about `7e-5`. With it, what remains is rounding
in float64 arithmetic, a few times `1e-16` around machine epsilon (the grey
line), and at more than half of the steps the total is exactly 1, which a
logarithmic axis cannot show.

In [ ]:
fig, ax = plt.subplots()
ax.axhline(np.finfo(float).eps, color=REFERENCE, lw=2.5, label="machine epsilon")
# A total of exactly 1 (zero drift) cannot be drawn on a log axis: leave it out.
ax.semilogy(ts_sir, np.where(resid_free > 0, resid_free, np.nan), color=palette[1], label="ODE only")
ax.semilogy(
    ts_sir, np.where(resid_cons > 0, resid_cons, np.nan), "o", ms=2.5,
    color=palette[0], label="ODE + conservation law",
)
ax.set_ylim(1e-17, 1e-2)
ax.set_xlabel("$t$ (days)")
ax.set_ylabel("$|S + I + R - 1|$")
ax.legend(loc="center right")
plt.show()

print(f"max drift without constraint: {resid_free.max():.2e}")
print(f"max drift with    constraint: {resid_cons.max():.2e}")

## Summary

A `Conservation(A, p)` object enforces the linear invariant `A @ x = p` as a
noiseless pseudo-observation. It is always active, so it rides along in
`ODEInformation(..., constraints=[...])` and reuses the Kalman update of the ODE
itself. With the SIR population sum constrained, the invariant held to float64
rounding (`~1e-16`) instead of drifting to `~7e-5`.

Data are the other kind of extra information. They exist only at some times, so
they enter as a separate observation model (`prepare_observations`, passed as
`obs_model=`); the [Inference from data](../parameter-estimation/) tutorials use
them to estimate parameters and hidden inputs. For more exotic needs, the library
also offers nonlinear observation models through `BlackBoxMeasurement` and
`TransformedMeasurement`, and Gaussian-process priors via `MaternPrior`. To pick
a linearization see [Linearization schemes](../../corrections/), and
[How to choose](../../how-to-choose/) to match a prior and step size to your
problem.